# 🧠 Module 3 — Employee Attrition Prediction
## TalentPulse | XGBoost + SMOTE + SHAP + DiCE

**Pipeline:**
1. Load IBM HR Dataset → Feature Engineering (2 new columns + 2 proxy features)
2. Train/Test Split → SMOTE on training set only
3. Stage 1: XGBoost (binary attrition Yes/No) + LightGBM comparison
4. Stage 2: Random Forest (reason classification — synthetic labels)
5. SHAP (TreeExplainer) → per-employee top-5 feature attributions
6. DiCE → 3 actionable counterfactuals per high-risk employee (>50%)
7. Save 4 CSVs + model files to `/kaggle/working/`

> **Note:** Module 2 `skill_gap_score` is not yet available.  
> A `burnout_proxy` feature is used as a substitute. The architecture is designed  
> so `skill_gap_score` can be added later as a column — no code changes needed.

## ⚙️ Step 0: Install Required Packages

In [ ]:
# Install packages not available by default on Kaggle
# dice-ml and imbalanced-learn need explicit install
import subprocess
subprocess.run(['pip', 'install', 'dice-ml', 'shap', 'imbalanced-learn', '-q'], check=True)
print("✅ Packages installed")

## 📦 Step 1: Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb

# Explainability
import shap
import dice_ml

# Plotting
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
np.random.seed(42)

# Paths
#DATA_PATH   = '/kaggle/input/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv'
DATA_PATH = '/kaggle/input/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv'
OUTPUT_DIR  = '/kaggle/working/'

print("✅ All libraries imported successfully")
print(f"   XGBoost version : {xgb.__version__}")
print(f"   LightGBM version: {lgb.__version__}")
print(f"   SHAP version    : {shap.__version__}")

## 📂 Step 2: Load Dataset

The IBM HR Analytics dataset is loaded directly from Kaggle input.  
- **1,470 employees | 35 columns | ~16% attrition rate**  
- Columns with zero variance (`EmployeeCount`, `Over18`, `StandardHours`) are dropped.  
- Redundant financial columns (`DailyRate`, `HourlyRate`, `MonthlyRate`) are dropped to avoid multicollinearity with `MonthlyIncome`.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {df_raw.shape}")

# Save EmployeeNumber separately — it's an ID, not a feature
employee_ids = df_raw['EmployeeNumber'].copy()

# Drop zero-variance + redundant columns
COLS_TO_DROP = [
    'EmployeeCount',   # Always 1
    'Over18',          # Always 'Y'
    'StandardHours',   # Always 80
    'DailyRate',       # Redundant with MonthlyIncome
    'HourlyRate',      # Redundant with MonthlyIncome
    'MonthlyRate',     # Redundant with MonthlyIncome
]
df = df_raw.drop(columns=COLS_TO_DROP).copy()
print(f"Shape after dropping useless columns: {df.shape}")
print(f"\nAttrition distribution:")
print(df['Attrition'].value_counts())
print(f"  Attrition rate: {(df['Attrition']=='Yes').mean():.1%}")

# Quick sanity check — IBM dataset has no missing values
missing = df.isnull().sum().sum()
print(f"\nMissing values: {missing} ✅" if missing == 0 else f"Missing values: {missing} ⚠️")

## 🔧 Step 3: Feature Engineering

### New columns created at runtime (no database needed):

| Feature | Formula | Meaning |
|---|---|---|
| `income_to_dept_avg` | `MonthlyIncome / dept_mean(MonthlyIncome)` | Relative pay position within department |
| `promotion_stagnation` | `YearsAtCompany - YearsSinceLastPromotion` | Years effectively "stuck" without promotion |
| `burnout_proxy` | `OverTime_bin × (4 - WorkLifeBalance)` | Combined overtime + poor WLB signal *(replaces skill_gap_score until Module 2 is ready)* |
| `total_satisfaction` | mean of 4 satisfaction columns | Aggregate wellbeing score |

In [ ]:
# ── Feature 1: income_to_dept_avg ──────────────────────────────────────────
dept_avg_series = df.groupby('Department')['MonthlyIncome'].transform('mean')
df['income_to_dept_avg'] = (df['MonthlyIncome'] / dept_avg_series).round(4)

# Save dept averages dict — needed for inference time in FastAPI
dept_avg_dict = df.groupby('Department')['MonthlyIncome'].mean().to_dict()

# ── Feature 2: promotion_stagnation ────────────────────────────────────────
df['promotion_stagnation'] = df['YearsAtCompany'] - df['YearsSinceLastPromotion']

# ── Feature 3: burnout_proxy (substitute for skill_gap_score from M2) ──────
# Temporarily encode OverTime for this calculation
ot_bin = df['OverTime'].map({'Yes': 1, 'No': 0})
df['burnout_proxy'] = ot_bin * (4 - df['WorkLifeBalance'])

# ── Feature 4: total_satisfaction ───────────────────────────────────────────
sat_cols = ['JobSatisfaction', 'EnvironmentSatisfaction',
            'RelationshipSatisfaction', 'WorkLifeBalance']
df['total_satisfaction'] = df[sat_cols].mean(axis=1).round(4)

print("✅ Engineered features created:")
for col in ['income_to_dept_avg', 'promotion_stagnation', 'burnout_proxy', 'total_satisfaction']:
    print(f"   {col:<30}  min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}")

## 🔤 Step 4: Encode Categorical Variables

All string columns are converted to numeric before model training.
Encodings are saved in `preprocessor.pkl` so inference can apply the **same** mappings.

In [ ]:
label_encoders = {}

# ── Binary (direct map) ─────────────────────────────────────────────────────
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df['Gender']    = df['Gender'].map({'Male': 1, 'Female': 0})
df['OverTime']  = df['OverTime'].map({'Yes': 1, 'No': 0})

# ── Ordinal (preserve order) ────────────────────────────────────────────────
business_travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
marital_status_map  = {'Single': 0, 'Married': 1, 'Divorced': 2}
df['BusinessTravel'] = df['BusinessTravel'].map(business_travel_map)
df['MaritalStatus']  = df['MaritalStatus'].map(marital_status_map)

# ── Label encode multi-category strings ─────────────────────────────────────
for col in ['JobRole', 'Department', 'EducationField']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le   # Save each encoder

# Sanity check — no strings left
remaining_str_cols = df.select_dtypes(include='object').columns.tolist()
if remaining_str_cols:
    print(f"⚠️  Still string: {remaining_str_cols}")
else:
    print("✅ All columns are numeric")

print(f"\nDataFrame shape: {df.shape}")
df.head(2)

## 🧩 Step 5: Prepare Feature Matrix

`EmployeeNumber` is excluded from features (it's an ID).  
`Attrition` is the target variable.

In [ ]:
FEATURES = [col for col in df.columns if col not in ['Attrition', 'EmployeeNumber']]
TARGET   = 'Attrition'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"Feature matrix  X: {X.shape}")
print(f"Target vector   y: {y.shape}  (attrition rate: {y.mean():.1%})")
print(f"\nAll {len(FEATURES)} features:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")

# Save feature list for consistent ordering at inference time
with open(f'{OUTPUT_DIR}feature_columns.json', 'w') as fp:
    json.dump(FEATURES, fp)
print(f"\n✅ feature_columns.json saved ({len(FEATURES)} features)")

## ⚖️ Step 6: Train/Test Split + SMOTE

**Critical:** SMOTE is applied **after** the split, to the **training set only**.  
Applying SMOTE before splitting causes data leakage (synthetic points from training contaminate the test set, inflating accuracy artificially).

- Split: 80% train / 20% test, stratified to preserve 16% attrition rate
- SMOTE sampling strategy: 0.5 (minority → 50% of majority) — avoids over-generating far from real data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Before SMOTE:")
print(f"  Train: {X_train.shape[0]} rows | Attrition rate: {y_train.mean():.1%}")
print(f"  Test : {X_test.shape[0]}  rows | Attrition rate: {y_test.mean():.1%}")

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (training set only):")
print(f"  Class 0 (Stay) : {(y_train_sm==0).sum()}")
print(f"  Class 1 (Leave): {(y_train_sm==1).sum()}")
print(f"  Total          : {len(y_train_sm)}")

## 🤖 Step 7a: Stage 1 — XGBoost Attrition Classifier

**Why XGBoost?** Consistently achieves AUC-ROC ~0.88–0.92 on this exact IBM dataset in published benchmarks.  
It is the de-facto SOTA for tabular HR classification tasks.

Key settings:
- `scale_pos_weight` = count(No)/count(Yes) ≈ 5.1 → additional class imbalance correction on top of SMOTE
- `subsample=0.8, colsample_bytree=0.8` → regularization via row/column sampling
- `eval_metric='logloss'` → optimises log-likelihood (appropriate for probability output)

In [ ]:
# scale_pos_weight: additional imbalance correction even after SMOTE
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    n_estimators      = 500,
    max_depth         = 5,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 3,
    gamma             = 0.1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    scale_pos_weight  = scale_pos_weight,
    eval_metric       = 'logloss',
    random_state      = 42,
    n_jobs            = -1
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set   = [(X_test, y_test)],
    verbose    = 100
)
print("\n✅ XGBoost training complete")

### 📊 Stage 1 Evaluation

In [ ]:
y_pred       = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=" * 62)
print("  STAGE 1 — XGBoost Binary Attrition Classifier")
print("=" * 62)
print(f"  Accuracy           : {accuracy_score(y_test, y_pred):.4f}")
print(f"  AUC-ROC            : {roc_auc_score(y_test, y_pred_proba):.4f}  ← primary metric")
print(f"  F1 (macro)         : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"  F1 (Attrition=Yes) : {f1_score(y_test, y_pred, average='binary'):.4f}")
print(f"  Precision (Yes)    : {precision_score(y_test, y_pred):.4f}")
print(f"  Recall    (Yes)    : {recall_score(y_test, y_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay', 'Leave']))
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")
print(f"  TN={cm[0,0]} | FP={cm[0,1]} | FN={cm[1,0]} | TP={cm[1,1]}")
print()
# A high FN means we're missing real leavers — important for HR context
fn_rate = cm[1,0] / (cm[1,0] + cm[1,1])
print(f"  False Negative Rate (missed leavers): {fn_rate:.1%}")

# ── LightGBM comparison ─────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators    = 300,
    max_depth       = 5,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    class_weight    = 'balanced',
    random_state    = 42,
    verbose         = -1
)
lgb_model.fit(X_train_sm, y_train_sm)
y_pred_lgb   = lgb_model.predict(X_test)
y_proba_lgb  = lgb_model.predict_proba(X_test)[:, 1]

print("\n" + "=" * 62)
print("  COMPARISON — LightGBM")
print("=" * 62)
print(f"  Accuracy           : {accuracy_score(y_test, y_pred_lgb):.4f}")
print(f"  AUC-ROC            : {roc_auc_score(y_test, y_proba_lgb):.4f}")
print(f"  F1 (Attrition=Yes) : {f1_score(y_test, y_pred_lgb, average='binary'):.4f}")

## 🏷️ Step 7b: Stage 2 — Attrition Reason Classifier

The IBM dataset has no "reason for leaving" column.  
We apply **deterministic rule-based labeling** to assign one of 4 reason categories  
to each training-set employee with `Attrition=Yes`, then train a Random Forest.

| Priority | Condition | Label |
|---|---|---|
| 1 | OverTime=1 **AND** WorkLifeBalance ≤ 2 | `Burnout` |
| 2 | income_to_dept_avg < 0.85 | `Compensation` |
| 3 | promotion_stagnation > 5 **AND** TrainingTimesLastYear ≤ 1 | `Stagnation` |
| 4 | JobLevel ≤ 2 **AND** YearsAtCompany > 5 **AND** YearsSinceLastPromotion > 3 | `Career Growth` |
| default | — | `Compensation` |

In [ ]:
REASON_MAP = {0: 'Burnout', 1: 'Compensation', 2: 'Stagnation', 3: 'Career Growth'}

def assign_reason(row):
    """Rule-based reason labeling for Attrition=Yes rows."""
    if row['OverTime'] == 1 and row['WorkLifeBalance'] <= 2:
        return 0  # Burnout
    if row['income_to_dept_avg'] < 0.85:
        return 1  # Compensation
    if row['promotion_stagnation'] > 5 and row['TrainingTimesLastYear'] <= 1:
        return 2  # Stagnation
    if (row['JobLevel'] <= 2 and row['YearsAtCompany'] > 5
            and row['YearsSinceLastPromotion'] > 3):
        return 3  # Career Growth
    return 1      # Default: Compensation

# Label training "Yes" rows
train_yes_df = X_train.copy()
train_yes_df['Attrition'] = y_train.values
train_yes_df = train_yes_df[train_yes_df['Attrition'] == 1].copy()
train_yes_df['reason'] = train_yes_df.apply(assign_reason, axis=1)

print("Reason distribution in training attrition cases:")
for r_id, r_name in REASON_MAP.items():
    n = (train_yes_df['reason'] == r_id).sum()
    print(f"  {r_name:<15} {n:3d}  ({n/len(train_yes_df):.1%})")

# Train Stage 2 Random Forest
rf_model = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 8,
    min_samples_leaf = 2,
    class_weight  = 'balanced',
    random_state  = 42,
    n_jobs        = -1
)
rf_model.fit(train_yes_df[FEATURES], train_yes_df['reason'])

# Evaluate on test "Yes" rows
test_yes_df = X_test.copy()
test_yes_df['Attrition'] = y_test.values
test_yes_df = test_yes_df[test_yes_df['Attrition'] == 1].copy()
test_yes_df['reason'] = test_yes_df.apply(assign_reason, axis=1)

y_pred_reason = rf_model.predict(test_yes_df[FEATURES])
acc_r  = accuracy_score(test_yes_df['reason'], y_pred_reason)
f1_r   = f1_score(test_yes_df['reason'], y_pred_reason, average='macro')
print(f"\n✅ Stage 2 metrics on test 'Yes' cases ({len(test_yes_df)} employees):")
print(f"   Accuracy : {acc_r:.4f}")
print(f"   F1 Macro : {f1_r:.4f}  (synthetic labels — expected ~0.65–0.80)")

## 📈 Step 8: Generate Predictions for All Employees

In [ ]:
X_all   = df[FEATURES].copy()
y_all   = df[TARGET].copy()

# Stage 1 predictions
all_proba = xgb_model.predict_proba(X_all)[:, 1]
all_pred  = xgb_model.predict(X_all)

# Stage 2 predictions
reason_proba_raw = rf_model.predict_proba(X_all)
reason_pred      = rf_model.predict(X_all)

# Map probabilities safely — RF may not have seen all 4 classes
reason_proba_full = np.zeros((len(X_all), 4))  # shape (n, 4) — always 4 cols
for col_idx, cls_label in enumerate(rf_model.classes_):
    reason_proba_full[:, int(cls_label)] = reason_proba_raw[:, col_idx]

def get_risk_tier(p):
    if p >= 0.70: return 'Critical'
    if p >= 0.50: return 'High'
    if p >= 0.30: return 'Medium'
    return 'Low'

results_df = pd.DataFrame({
    'EmployeeNumber'           : df_raw['EmployeeNumber'].values,
    'Age'                      : df_raw['Age'].values,
    'Department'               : df_raw['Department'].values,
    'JobRole'                  : df_raw['JobRole'].values,
    'MonthlyIncome'            : df_raw['MonthlyIncome'].values,
    'attrition_verdict'        : ['Yes' if p == 1 else 'No' for p in all_pred],
    'attrition_probability'    : all_proba.round(4),
    'risk_tier'                : [get_risk_tier(p) for p in all_proba],
    'primary_reason'           : [REASON_MAP[reason_pred[i]] if all_pred[i] == 1 else 'N/A'
                                   for i in range(len(all_pred))],
    'reason_prob_burnout'      : reason_proba_full[:, 0].round(4),
    'reason_prob_compensation' : reason_proba_full[:, 1].round(4),
    'reason_prob_stagnation'   : reason_proba_full[:, 2].round(4),
    'reason_prob_career_growth': reason_proba_full[:, 3].round(4),
})

# # Stage 2 predictions (reason probabilities for ALL employees)
# # The frontend hides reason data for "No" predictions, but we compute for completeness
# reason_proba = rf_model.predict_proba(X_all)   # shape (n, 4)
# reason_pred  = rf_model.predict(X_all)

# def get_risk_tier(p):
#     if p >= 0.70: return 'Critical'
#     if p >= 0.50: return 'High'
#     if p >= 0.30: return 'Medium'
#     return 'Low'

# # Build the main results dataframe
# results_df = pd.DataFrame({
#     'EmployeeNumber'       : df_raw['EmployeeNumber'].values,
#     'Age'                  : df_raw['Age'].values,
#     'Department'           : df_raw['Department'].values,
#     'JobRole'              : df_raw['JobRole'].values,
#     'MonthlyIncome'        : df_raw['MonthlyIncome'].values,
#     'attrition_verdict'    : ['Yes' if p == 1 else 'No' for p in all_pred],
#     'attrition_probability': all_proba.round(4),
#     'risk_tier'            : [get_risk_tier(p) for p in all_proba],
#     'primary_reason'       : [REASON_MAP[reason_pred[i]] if all_pred[i] == 1 else 'N/A'
#                                for i in range(len(all_pred))],
#     'reason_prob_burnout'      : reason_proba[:, 0].round(4),
#     'reason_prob_compensation' : reason_proba[:, 1].round(4),
#     'reason_prob_stagnation'   : reason_proba[:, 2].round(4),
#     'reason_prob_career_growth': reason_proba[:, 3].round(4),
# })

print(f"Results dataframe: {results_df.shape}")
print(f"\nRisk tier distribution:")
print(results_df['risk_tier'].value_counts().to_string())
print(f"\nPrimary reason (for predicted-Yes employees):")
print(results_df[results_df['attrition_verdict']=='Yes']['primary_reason'].value_counts().to_string())
results_df.head(5)




## 🔍 Step 9: SHAP Analysis (TreeExplainer)

`TreeExplainer` is the fast, exact algorithm for tree-based models.  
It returns a matrix of shape `(n_employees, n_features)` where each value is  
the SHAP contribution of that feature to that employee's attrition probability.

- Positive SHAP value → pushes prediction toward **Attrition=Yes**
- Negative SHAP value → pushes prediction toward **Attrition=No**

In [ ]:
print("Computing SHAP values for all employees...")
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_all)
base_value  = float(explainer.expected_value)

print(f"✅ SHAP complete")
print(f"   SHAP matrix shape : {shap_values.shape}")
print(f"   Base value (avg)  : {base_value:.4f}  (~dataset attrition rate in log-odds space)")

# ── Global feature importance (mean |SHAP|) ──────────────────────────────────
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES
).sort_values(ascending=False)

print("\nTop 15 Global Feature Importances (mean |SHAP| across all employees):")
for feat, val in mean_abs_shap.head(15).items():
    bar = '█' * int(val * 300)
    print(f"  {feat:<35} {val:.4f}  {bar}")

# ── Save global SHAP summary plot ─────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_all, plot_type='bar',
                  max_display=15, show=False)
plt.title('Global Feature Importance (Mean |SHAP|)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}shap_global_importance.png', dpi=100, bbox_inches='tight')
plt.close()
print("\n✅ shap_global_importance.png saved")

# ── Build per-employee SHAP dataframe ────────────────────────────────────────
shap_records = []
for i in range(len(X_all)):
    emp_shap   = shap_values[i]
    sorted_idx = np.argsort(np.abs(emp_shap))[::-1]
    record = {
        'EmployeeNumber': int(df_raw['EmployeeNumber'].iloc[i]),
        'base_value'    : round(base_value, 6)
    }
    for rank in range(5):
        fi = sorted_idx[rank]
        record[f'shap_feature_{rank+1}'] = FEATURES[fi]
        record[f'shap_value_{rank+1}']   = round(float(emp_shap[fi]), 6)
    shap_records.append(record)

shap_df = pd.DataFrame(shap_records)
print(f"SHAP results dataframe: {shap_df.shape}")
shap_df.head(3)

## 💡 Step 10: DiCE Counterfactual Interventions

DiCE (Diverse Counterfactual Explanations) answers:  
*"What is the **minimum actionable change** that would flip this employee from 'at risk' to 'safe'?"*

**Scope:** Only employees with `attrition_probability > 0.50` (top risk tier) receive interventions.  
**Per employee:** 3 diverse counterfactual plans are generated.  
**Immutable features:** Age, Gender, TotalWorkingYears, YearsAtCompany, Education, NumCompaniesWorked  
**Mutable features:** OverTime, MonthlyIncome, JobLevel, WorkLifeBalance, JobSatisfaction, and others HR can act on.

In [ ]:
# # ── Identify high-risk employees ─────────────────────────────────────────────
# high_risk_mask    = all_proba > 0.50
# high_risk_indices = np.where(high_risk_mask)[0]
# print(f"Employees with attrition_probability > 50%: {len(high_risk_indices)}")

# # Cap at top 150 to keep runtime under ~20 minutes
# MAX_DICE = 150
# if len(high_risk_indices) > MAX_DICE:
#     top_idx           = np.argsort(all_proba)[::-1][:MAX_DICE]
#     high_risk_indices = top_idx
#     print(f"Capped to top {MAX_DICE} highest-risk employees for runtime")

# # ── DiCE setup ────────────────────────────────────────────────────────────────
# # Background data: use original training set (not SMOTE-augmented)
# train_bg = X_train.copy()
# train_bg['Attrition'] = y_train.values

# # Continuous features (DiCE needs to know which are continuous vs categorical)
# continuous_features = [
#     'Age', 'DistanceFromHome', 'MonthlyIncome', 'NumCompaniesWorked',
#     'PercentSalaryHike', 'TotalWorkingYears', 'TrainingTimesLastYear',
#     'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
#     'YearsWithCurrManager', 'income_to_dept_avg', 'promotion_stagnation',
#     'burnout_proxy', 'total_satisfaction'
# ]
# continuous_features = [f for f in continuous_features if f in FEATURES]

# d_dice = dice_ml.Data(
#     dataframe         = train_bg,
#     continuous_features = continuous_features,
#     outcome_name      = 'Attrition'
# )
# m_dice = dice_ml.Model(model=xgb_model, backend='sklearn')
# exp_dice = dice_ml.Dice(d_dice, m_dice, method='random')

# # Mutable features — HR can actually change these
# mutable_features = [
#     f for f in [
#         'OverTime', 'MonthlyIncome', 'YearsSinceLastPromotion',
#         'JobLevel', 'WorkLifeBalance', 'JobSatisfaction',
#         'EnvironmentSatisfaction', 'StockOptionLevel',
#         'TrainingTimesLastYear', 'DistanceFromHome'
#     ] if f in FEATURES
# ]

# # Global permitted ranges (based on IBM dataset bounds)
# permitted_range = {
#     'MonthlyIncome'          : [int(df['MonthlyIncome'].min()), int(df['MonthlyIncome'].quantile(0.95))],
#     'JobLevel'               : [1, 5],
#     'StockOptionLevel'       : [0, 3],
#     'WorkLifeBalance'        : [1, 4],
#     'JobSatisfaction'        : [1, 4],
#     'EnvironmentSatisfaction': [1, 4],
#     'TrainingTimesLastYear'  : [0, 6],
#     'YearsSinceLastPromotion': [0, 15],
#     'OverTime'               : [0, 1],
#     'DistanceFromHome'       : [1, 29],
# }
# permitted_range = {k: v for k, v in permitted_range.items() if k in mutable_features}

# def make_label(feat, cur, sug):
#     """Human-readable intervention label."""
#     t = {
#         'OverTime'               : f"{'Remove' if sug < cur else 'Adjust'} overtime requirement",
#         'MonthlyIncome'          : f"Salary raise: ${cur:,.0f} → ${sug:,.0f}/month",
#         'YearsSinceLastPromotion': f"Promote employee (reset promotion clock)",
#         'JobLevel'               : f"Promote to Level {int(sug)}",
#         'WorkLifeBalance'        : f"Improve work-life balance programs",
#         'JobSatisfaction'        : f"Improve job satisfaction (role enrichment / feedback)",
#         'EnvironmentSatisfaction': f"Improve work environment conditions",
#         'StockOptionLevel'       : f"Grant stock options (Level {int(sug)})",
#         'TrainingTimesLastYear'  : f"Increase training to {int(sug)} sessions/year",
#         'DistanceFromHome'       : f"Offer remote/hybrid work arrangement",
#     }
#     return t.get(feat, f"Adjust {feat}: {cur:.1f} → {sug:.1f}")

# print(f"\n✅ DiCE setup complete")
# print(f"   Mutable features  : {mutable_features}")
# print(f"   Continuous feats  : {len(continuous_features)}")



















# # ── Identify high-risk employees ─────────────────────────────────────────────
# high_risk_mask    = all_proba > 0.50
# high_risk_indices = np.where(high_risk_mask)[0]
# print(f"Employees with attrition_probability > 50%: {len(high_risk_indices)}")

# MAX_DICE = 150
# if len(high_risk_indices) > MAX_DICE:
#     top_idx           = np.argsort(all_proba)[::-1][:MAX_DICE]
#     high_risk_indices = top_idx
#     print(f"Capped to top {MAX_DICE} highest-risk employees")

# # ── DiCE setup ────────────────────────────────────────────────────────────────
# train_bg = X_train.copy()
# train_bg['Attrition'] = y_train.values.astype(int)

# continuous_features = [f for f in [
#     'Age', 'DistanceFromHome', 'MonthlyIncome', 'NumCompaniesWorked',
#     'PercentSalaryHike', 'TotalWorkingYears', 'TrainingTimesLastYear',
#     'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
#     'YearsWithCurrManager', 'income_to_dept_avg', 'promotion_stagnation',
#     'burnout_proxy', 'total_satisfaction'
# ] if f in FEATURES]

# d_dice    = dice_ml.Data(dataframe=train_bg, continuous_features=continuous_features, outcome_name='Attrition')
# m_dice    = dice_ml.Model(model=xgb_model, backend='sklearn')
# exp_dice  = dice_ml.Dice(d_dice, m_dice, method='genetic')   # 'genetic' is more stable than 'random'

# mutable_features = [f for f in [
#     'OverTime', 'MonthlyIncome', 'YearsSinceLastPromotion',
#     'JobLevel', 'WorkLifeBalance', 'JobSatisfaction',
#     'EnvironmentSatisfaction', 'StockOptionLevel',
#     'TrainingTimesLastYear', 'DistanceFromHome'
# ] if f in FEATURES]

# permitted_range = {
#     'MonthlyIncome'          : [int(df['MonthlyIncome'].min()), int(df['MonthlyIncome'].quantile(0.95))],
#     'JobLevel'               : [1, 5],
#     'StockOptionLevel'       : [0, 3],
#     'WorkLifeBalance'        : [1, 4],
#     'JobSatisfaction'        : [1, 4],
#     'EnvironmentSatisfaction': [1, 4],
#     'TrainingTimesLastYear'  : [0, 6],
#     'YearsSinceLastPromotion': [0, 15],
#     'OverTime'               : [0, 1],
#     'DistanceFromHome'       : [1, 29],
# }
# permitted_range = {k: v for k, v in permitted_range.items() if k in mutable_features}

# def make_label(feat, cur, sug):
#     t = {
#         'OverTime'               : f"{'Remove' if sug < cur else 'Adjust'} overtime requirement",
#         'MonthlyIncome'          : f"Salary raise: ${cur:,.0f} → ${sug:,.0f}/month",
#         'YearsSinceLastPromotion': f"Promote employee (reset promotion clock)",
#         'JobLevel'               : f"Promote to Level {int(sug)}",
#         'WorkLifeBalance'        : f"Improve work-life balance programs",
#         'JobSatisfaction'        : f"Improve job satisfaction (role enrichment / feedback)",
#         'EnvironmentSatisfaction': f"Improve work environment conditions",
#         'StockOptionLevel'       : f"Grant stock options (Level {int(sug)})",
#         'TrainingTimesLastYear'  : f"Increase training to {int(sug)} sessions/year",
#         'DistanceFromHome'       : f"Offer remote/hybrid work arrangement",
#     }
#     return t.get(feat, f"Adjust {feat}: {cur:.1f} → {sug:.1f}")

# print("✅ DiCE setup complete")
# print(f"   Method          : genetic")
# print(f"   Mutable features: {mutable_features}")



# ══════════════════════════════════════════════════════════════════════════════
# Step 10: Custom Counterfactual Intervention Engine
# (Replaces DiCE — direct XGBoost perturbation, guaranteed to produce results)
#
# For each high-risk employee, we scan every mutable feature across its
# allowed range, find the top single-feature changes by risk reduction,
# then combine the best 2 into a "combo" intervention — giving 3 CFs total.
# ══════════════════════════════════════════════════════════════════════════════













high_risk_mask    = all_proba > 0.50
high_risk_indices = np.where(high_risk_mask)[0]
print(f"High-risk employees (>50%): {len(high_risk_indices)}")

# ── Mutable feature scan ranges ───────────────────────────────────────────────
MUTABLE_RANGES = {
    'OverTime'               : [0, 1],
    'JobSatisfaction'        : [1, 2, 3, 4],
    'EnvironmentSatisfaction': [1, 2, 3, 4],
    'WorkLifeBalance'        : [1, 2, 3, 4],
    'StockOptionLevel'       : [0, 1, 2, 3],
    'JobLevel'               : [1, 2, 3, 4, 5],
    'YearsSinceLastPromotion': [0, 1, 2],
    'TrainingTimesLastYear'  : [2, 3, 4, 5, 6],
    'MonthlyIncome'          : None,   # continuous — handled separately
    'DistanceFromHome'       : None,
}
MUTABLE_FEATURES = [f for f in MUTABLE_RANGES if f in FEATURES]

def get_scan_values(feat, current_val, df_col):
    """Return candidate values to try for a given feature."""
    if MUTABLE_RANGES[feat] is not None:
        # Discrete: only return values that are BETTER than current
        if feat == 'OverTime':
            return [0] if current_val > 0 else []
        candidates = [v for v in MUTABLE_RANGES[feat] if v > current_val]
        if feat == 'YearsSinceLastPromotion':
            candidates = [v for v in MUTABLE_RANGES[feat] if v < current_val]
        return candidates
    else:
        # Continuous: try percentile steps upward
        p50 = df_col.quantile(0.50)
        p75 = df_col.quantile(0.75)
        p90 = df_col.quantile(0.90)
        if feat == 'DistanceFromHome':
            p25 = df_col.quantile(0.25)
            return [v for v in [1, p25, p50] if v < current_val]
        return [v for v in [p50, p75, p90] if v > current_val]

def make_label(feat, cur, sug):
    labels = {
        'OverTime'               : "Remove overtime requirement",
        # 'MonthlyIncome'          : f"Salary raise: ${cur:,.0f} → ${sug:,.0f}/month (+{(sug/cur-1)*100:.0f}%)",
        'MonthlyIncome' : f"Salary raise: ${cur:,.0f} → ${sug:,.0f}/month" + (f" (+{(sug/cur-1)*100:.0f}%)" if cur > 0 else ""),
        'YearsSinceLastPromotion': "Promote employee (reset promotion clock to 0)",
        'JobLevel'               : f"Promote to Job Level {int(sug)}",
        'WorkLifeBalance'        : f"Improve work-life balance: {int(cur)} → {int(sug)}/4",
        'JobSatisfaction'        : f"Improve job satisfaction: {int(cur)} → {int(sug)}/4",
        'EnvironmentSatisfaction': f"Improve work environment: {int(cur)} → {int(sug)}/4",
        'StockOptionLevel'       : f"Grant stock options (Level {int(cur)} → {int(sug)})",
        'TrainingTimesLastYear'  : f"Increase training sessions: {int(cur)} → {int(sug)}/year",
        'DistanceFromHome'       : f"Offer remote/hybrid work (commute: {cur:.0f} → {sug:.0f} miles)",
    }
    return labels.get(feat, f"Change {feat}: {cur:.2f} → {sug:.2f}")

# ── Main loop ─────────────────────────────────────────────────────────────────
dice_records = []
print("Generating interventions... ", end='', flush=True)

for loop_num, emp_idx in enumerate(high_risk_indices):
    if loop_num % 30 == 0:
        print(f"{loop_num}", end='..', flush=True)

    emp_row      = X_all.iloc[[emp_idx]].copy()
    emp_id       = int(df_raw['EmployeeNumber'].iloc[emp_idx])
    current_risk = float(all_proba[emp_idx])

    # ── CF 1 & 2: Best and 2nd-best single-feature changes ───────────────────
    single_changes = []
    for feat in MUTABLE_FEATURES:
        cur_val   = float(emp_row[feat].iloc[0])
        scan_vals = get_scan_values(feat, cur_val, df[feat])
        for try_val in scan_vals:
            perturbed        = emp_row.copy()
            perturbed[feat]  = try_val
            new_risk         = float(xgb_model.predict_proba(perturbed)[0, 1])
            reduction        = current_risk - new_risk
            if reduction > 0.01:   # only record meaningful improvements
                single_changes.append((reduction, feat, cur_val, try_val, new_risk))

    single_changes.sort(reverse=True)

    # CF index 1: best single change
    if single_changes:
        red, feat, cur, sug, new_risk = single_changes[0]
        dice_records.append({
            'EmployeeNumber'     : emp_id,
            'cf_index'           : 1,
            'feature_changed'    : feat,
            'current_value'      : round(cur, 4),
            'suggested_value'    : round(float(sug), 4),
            'new_attrition_prob' : round(new_risk, 4),
            'risk_reduction'     : round(red, 4),
            'intervention_label' : make_label(feat, cur, sug),
        })

    # CF index 2: second-best single change (different feature from CF1)
    best_feat = single_changes[0][1] if single_changes else None
    for red, feat, cur, sug, new_risk in single_changes[1:]:
        if feat != best_feat:
            dice_records.append({
                'EmployeeNumber'     : emp_id,
                'cf_index'           : 2,
                'feature_changed'    : feat,
                'current_value'      : round(cur, 4),
                'suggested_value'    : round(float(sug), 4),
                'new_attrition_prob' : round(new_risk, 4),
                'risk_reduction'     : round(red, 4),
                'intervention_label' : make_label(feat, cur, sug),
            })
            break

    # CF index 3: combo — apply top 2 changes together
    top2 = [c for c in single_changes[:6]]
    if len(top2) >= 2:
        combo = emp_row.copy()
        used, labels_used = [], []
        for red, feat, cur, sug, _ in top2[:4]:
            if feat not in [u[0] for u in used]:
                combo[feat] = sug
                used.append((feat, cur, sug))
            if len(used) == 2:
                break
        if len(used) == 2:
            combo_risk = float(xgb_model.predict_proba(combo)[0, 1])
            for feat, cur, sug in used:
                dice_records.append({
                    'EmployeeNumber'     : emp_id,
                    'cf_index'           : 3,
                    'feature_changed'    : feat,
                    'current_value'      : round(cur, 4),
                    'suggested_value'    : round(float(sug), 4),
                    'new_attrition_prob' : round(combo_risk, 4),
                    'risk_reduction'     : round(current_risk - combo_risk, 4),
                    'intervention_label' : make_label(feat, cur, sug),
                })

print(f"\n\n✅ Done")
print(f"   Employees processed   : {len(high_risk_indices)}")
print(f"   Intervention rows     : {len(dice_records)}")

dice_df = pd.DataFrame(dice_records) if dice_records else pd.DataFrame(
    columns=['EmployeeNumber','cf_index','feature_changed','current_value',
             'suggested_value','new_attrition_prob','risk_reduction','intervention_label'])

if len(dice_df) > 0:
    print(f"\nMost common recommended interventions:")
    print(dice_df['feature_changed'].value_counts().head(8).to_string())
    print(f"\nMean risk reduction per intervention type:")
    print(dice_df.groupby('feature_changed')['risk_reduction'].mean()
          .sort_values(ascending=False).head(8).to_string())

In [ ]:
# # ── Generate counterfactuals (main loop) ─────────────────────────────────────
# dice_records = []
# failed_count = 0

# print(f"Generating counterfactuals for {len(high_risk_indices)} employees...")
# print("(This is the slow step — ~10–20 min on Kaggle CPU)")
# print("Progress: ", end='', flush=True)

# for loop_num, emp_idx in enumerate(high_risk_indices):
#     if loop_num % 20 == 0:
#         print(f"{loop_num}", end='..', flush=True)

#     try:
#         query_instance = X_all.iloc[[emp_idx]]
#         emp_id         = int(df_raw['EmployeeNumber'].iloc[emp_idx])
#         current_risk   = float(all_proba[emp_idx])

#         cf_result = exp_dice.generate_counterfactuals(
#             query_instances  = query_instance,
#             total_CFs        = 3,
#             desired_class    = "opposite",
#             features_to_vary = mutable_features,
#             permitted_range  = permitted_range
#         )

#         cf_df = cf_result.cf_examples_list[0].final_cfs_df
#         if cf_df is None or len(cf_df) == 0:
#             failed_count += 1
#             continue

#         for cf_num, (_, cf_row) in enumerate(cf_df.iterrows()):
#             # Apply ALL changes in this CF at once to get the joint new risk
#             cf_features = query_instance.copy()
#             changed_feats = []
#             for feat in mutable_features:
#                 if feat not in cf_row.index:
#                     continue
#                 orig = float(query_instance[feat].iloc[0])
#                 sugg = float(cf_row[feat])
#                 if abs(orig - sugg) > 0.01:
#                     cf_features[feat] = sugg
#                     changed_feats.append((feat, orig, sugg))

#             if not changed_feats:
#                 continue

#             # New risk = model output after ALL changes in this CF
#             new_risk = float(xgb_model.predict_proba(cf_features)[0, 1])

#             for feat, orig, sugg in changed_feats:
#                 dice_records.append({
#                     'EmployeeNumber'     : emp_id,
#                     'cf_index'           : cf_num + 1,
#                     'feature_changed'    : feat,
#                     'current_value'      : round(orig, 4),
#                     'suggested_value'    : round(sugg, 4),
#                     'new_attrition_prob' : round(new_risk, 4),
#                     'risk_reduction'     : round(current_risk - new_risk, 4),
#                     'intervention_label' : make_label(feat, orig, sugg)
#                 })

#     except Exception:
#         failed_count += 1
#         continue

# print(f"\n\n✅ DiCE complete")
# print(f"   Intervention rows generated : {len(dice_records)}")
# print(f"   Employees processed         : {len(high_risk_indices) - failed_count}")
# print(f"   Failed / skipped            : {failed_count}")

# dice_df = pd.DataFrame(dice_records) if dice_records else pd.DataFrame(
#     columns=['EmployeeNumber', 'cf_index', 'feature_changed',
#              'current_value', 'suggested_value', 'new_attrition_prob',
#              'risk_reduction', 'intervention_label']
# )
# if len(dice_df) > 0:
#     print(f"\nTop interventions by mean risk reduction:")
#     top = (dice_df.groupby('feature_changed')['risk_reduction']
#            .mean().sort_values(ascending=False).head(8))
#     print(top.to_string())















# # ── Generate counterfactuals ──────────────────────────────────────────────────
# dice_records = []
# failed_count = 0
# first_error_printed = False

# print(f"Generating counterfactuals for {len(high_risk_indices)} employees...")
# print("Progress: ", end='', flush=True)

# for loop_num, emp_idx in enumerate(high_risk_indices):
#     if loop_num % 20 == 0:
#         print(f"{loop_num}", end='..', flush=True)

#     try:
#         query_instance = X_all.iloc[[emp_idx]]
#         emp_id         = int(df_raw['EmployeeNumber'].iloc[emp_idx])
#         current_risk   = float(all_proba[emp_idx])

#         cf_result = exp_dice.generate_counterfactuals(
#             query_instances  = query_instance,
#             total_CFs        = 3,
#             desired_class    = 0,               # ← FIXED: 0 = "Stay", not "opposite"
#             features_to_vary = mutable_features,
#             permitted_range  = permitted_range
#         )

#         cf_df = cf_result.cf_examples_list[0].final_cfs_df
#         if cf_df is None or len(cf_df) == 0:
#             failed_count += 1
#             continue

#         for cf_num, (_, cf_row) in enumerate(cf_df.iterrows()):
#             cf_features   = query_instance.copy()
#             changed_feats = []
#             for feat in mutable_features:
#                 if feat not in cf_row.index:
#                     continue
#                 orig = float(query_instance[feat].iloc[0])
#                 sugg = float(cf_row[feat])
#                 if abs(orig - sugg) > 0.01:
#                     cf_features[feat] = sugg
#                     changed_feats.append((feat, orig, sugg))

#             if not changed_feats:
#                 continue

#             new_risk = float(xgb_model.predict_proba(cf_features)[0, 1])

#             for feat, orig, sugg in changed_feats:
#                 dice_records.append({
#                     'EmployeeNumber'     : emp_id,
#                     'cf_index'           : cf_num + 1,
#                     'feature_changed'    : feat,
#                     'current_value'      : round(orig, 4),
#                     'suggested_value'    : round(sugg, 4),
#                     'new_attrition_prob' : round(new_risk, 4),
#                     'risk_reduction'     : round(current_risk - new_risk, 4),
#                     'intervention_label' : make_label(feat, orig, sugg)
#                 })

#     except Exception as e:
#         failed_count += 1
#         if not first_error_printed:
#             print(f"\n⚠️  First error (showing once): {type(e).__name__}: {e}")
#             first_error_printed = True
#         continue

# print(f"\n\n✅ DiCE complete")
# print(f"   Rows generated  : {len(dice_records)}")
# print(f"   Successful      : {len(high_risk_indices) - failed_count}")
# print(f"   Failed/skipped  : {failed_count}")

# dice_df = pd.DataFrame(dice_records) if dice_records else pd.DataFrame(
#     columns=['EmployeeNumber','cf_index','feature_changed','current_value',
#              'suggested_value','new_attrition_prob','risk_reduction','intervention_label'])

# if len(dice_df) > 0:
#     print(f"\nTop interventions by mean risk reduction:")
#     print(dice_df.groupby('feature_changed')['risk_reduction'].mean().sort_values(ascending=False).head(8).to_string())

## 📊 Step 11: Build Model Metrics Table

In [ ]:
metrics_rows = []

def add_metric(name, value, model, notes=''):
    metrics_rows.append({'metric_name': name, 'value': round(float(value), 4),
                         'model': model, 'notes': notes})

# XGBoost Stage 1
add_metric('accuracy',          accuracy_score(y_test, y_pred),               'XGBoost_Stage1', 'Overall accuracy on held-out test set')
add_metric('auc_roc',           roc_auc_score(y_test, y_pred_proba),          'XGBoost_Stage1', 'PRIMARY metric — handles class imbalance')
add_metric('f1_macro',          f1_score(y_test, y_pred, average='macro'),    'XGBoost_Stage1', 'Balanced F1 across both classes')
add_metric('f1_minority',       f1_score(y_test, y_pred, average='binary'),   'XGBoost_Stage1', 'F1 for Attrition=Yes (minority class)')
add_metric('precision_minority',precision_score(y_test, y_pred),              'XGBoost_Stage1', 'Precision for Attrition=Yes')
add_metric('recall_minority',   recall_score(y_test, y_pred),                 'XGBoost_Stage1', 'Recall for Attrition=Yes')

# LightGBM comparison
add_metric('accuracy',    accuracy_score(y_test, y_pred_lgb),              'LightGBM_Compare', 'Comparison model')
add_metric('auc_roc',     roc_auc_score(y_test, y_proba_lgb),             'LightGBM_Compare', 'Comparison model')
add_metric('f1_minority', f1_score(y_test, y_pred_lgb, average='binary'), 'LightGBM_Compare', 'Comparison model')

# Stage 2
add_metric('accuracy', acc_r, 'RF_Stage2_Reason', 'Reason classification on test Yes-cases (synthetic labels)')
add_metric('f1_macro',  f1_r, 'RF_Stage2_Reason', 'Macro F1 across 4 reasons')

# SHAP
add_metric('mean_abs_shap_top_feature', float(mean_abs_shap.iloc[0]),
           'SHAP', f'Top feature: {mean_abs_shap.index[0]}')
add_metric('mean_abs_shap_global', float(np.abs(shap_values).mean()),
           'SHAP', 'Average |SHAP| across all features and employees')

# DiCE
if len(dice_df) > 0:
    add_metric('avg_risk_reduction_per_cf',
               dice_df.groupby(['EmployeeNumber','cf_index'])['risk_reduction'].first().mean(),
               'DiCE', 'Average attrition probability reduction per counterfactual set')
    add_metric('employees_with_interventions',
               dice_df['EmployeeNumber'].nunique(),
               'DiCE', 'High-risk employees with at least one counterfactual')

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string(index=False))

## 💾 Step 12: Save All Outputs

In [ ]:
print("Saving all output files to /kaggle/working/ ...\n")

# 1. Attrition results (main prediction CSV)
results_df.to_csv(f'{OUTPUT_DIR}attrition_results.csv', index=False)
print(f"✅ attrition_results.csv       — {len(results_df):,} rows")

# 2. SHAP per-employee top-5 features
shap_df.to_csv(f'{OUTPUT_DIR}shap_results.csv', index=False)
print(f"✅ shap_results.csv            — {len(shap_df):,} rows")

# 3. DiCE counterfactual interventions
dice_df.to_csv(f'{OUTPUT_DIR}dice_interventions.csv', index=False)
print(f"✅ dice_interventions.csv      — {len(dice_df):,} rows")

# 4. Model metrics summary
metrics_df.to_csv(f'{OUTPUT_DIR}model_metrics.csv', index=False)
print(f"✅ model_metrics.csv           — {len(metrics_df):,} rows")

# 5. XGBoost — native JSON (most portable, works with any XGBoost version)
xgb_model.save_model(f'{OUTPUT_DIR}model_xgb_stage1.json')
print(f"✅ model_xgb_stage1.json       — XGBoost native format")

# 6. XGBoost — pkl (sklearn interface, used by FastAPI)
joblib.dump(xgb_model, f'{OUTPUT_DIR}model_xgb_stage1.pkl')
print(f"✅ model_xgb_stage1.pkl        — scikit-learn compatible")

# 7. Random Forest Stage 2
joblib.dump(rf_model, f'{OUTPUT_DIR}model_rf_stage2.pkl')
print(f"✅ model_rf_stage2.pkl         — reason classifier")

# 8. Preprocessor bundle (everything inference needs)
preprocessor_bundle = {
    'label_encoders'     : label_encoders,
    'dept_avg_dict'      : dept_avg_dict,
    'business_travel_map': business_travel_map,
    'marital_status_map' : marital_status_map,
    'reason_map'         : REASON_MAP,
    'feature_columns'    : FEATURES,
}
joblib.dump(preprocessor_bundle, f'{OUTPUT_DIR}preprocessor.pkl')
print(f"✅ preprocessor.pkl            — encoders + dept avgs + feature list")

# 9. Feature columns JSON (re-saved here as authoritative copy)
with open(f'{OUTPUT_DIR}feature_columns.json', 'w') as fp:
    json.dump(FEATURES, fp)
print(f"✅ feature_columns.json        — {len(FEATURES)} features in order")

## ✅ Final Summary

In [ ]:
print("=" * 70)
print("  MODULE 3 — KAGGLE RUN COMPLETE")
print("=" * 70)

print(f"\n📊 DATASET")
print(f"   Employees     : {len(df):,}")
print(f"   Attrition rate: {y.mean():.1%}")
print(f"   Features used : {len(FEATURES)}")

print(f"\n🤖 MODEL PERFORMANCE (XGBoost Stage 1)")
print(f"   AUC-ROC          : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"   Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
print(f"   F1 (Attrition=Yes): {f1_score(y_test, y_pred, average='binary'):.4f}")

print(f"\n🎯 RISK SUMMARY (full dataset predictions)")
print(results_df['risk_tier'].value_counts().to_string())
print(f"   High-risk  (>50%): {(all_proba > 0.50).sum()}")
print(f"   Critical   (>70%): {(all_proba > 0.70).sum()}")

print(f"\n📁 OUTPUT FILES")
all_files = [
    'attrition_results.csv', 'shap_results.csv',
    'dice_interventions.csv', 'model_metrics.csv',
    'model_xgb_stage1.json', 'model_xgb_stage1.pkl',
    'model_rf_stage2.pkl', 'preprocessor.pkl',
    'feature_columns.json', 'shap_global_importance.png'
]
for fname in all_files:
    fpath = f'{OUTPUT_DIR}{fname}'
    if os.path.exists(fpath):
        kb = os.path.getsize(fpath) / 1024
        print(f"   ✅  {fname:<40} {kb:6.1f} KB")
    else:
        print(f"   ❌  {fname} — NOT FOUND")

print("\n🚀 Download files from Output section → use in FastAPI + HuggingFace Spaces")